# Análisis del Tráfico Aéreo en el Aeropuerto Internacional de San Francisco (SFO)
Este proyecto tiene como objetivo analizar el tráfico aéreo en el Aeropuerto Internacional de San Francisco (SFO) utilizando un conjunto de datos real. Abarca desde la preparación de datos hasta la creación de modelos predictivos y la presentación de resultados.

# Fase 1: Análisis Exploratorio y Preparación de Datos
En esta fase, cargaremos y limpiaremos los datos, realizaremos un análisis descriptivo y crearemos visualizaciones iniciales.

In [ ]:
# Importar bibliotecas necesarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset
data = pd.read_csv('air_traffic_data.csv')

# Mostrar las primeras filas del dataset
data.head()

In [ ]:
git config pull.rebase false

In [ ]:
# Limpieza de datos
# Manejo de valores nulos
data = data.dropna()

# Corrección de tipos de datos
data['Date'] = pd.to_datetime(data['Date'])

# Estandarización de nombres de columnas
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# Verificar los cambios
data.info()

In [ ]:
# Análisis descriptivo
# Top 10 aerolíneas con más pasajeros
top_airlines = data.groupby('operating_airline')['passenger_count'].sum().nlargest(10)
print(top_airlines)

# Top 10 rutas más transitadas
top_routes = data.groupby('geo_region')['passenger_count'].sum().nlargest(10)
print(top_routes)

In [ ]:
# Visualización: Distribución de pasajeros por terminal
sns.barplot(data=data, x='terminal', y='passenger_count', estimator=sum)
plt.title('Distribución de Pasajeros por Terminal')
plt.show()

In [ ]:
# Diagrama de caja: Distribución de pasajeros por tipo de vuelo
sns.boxplot(data=data, x='activity_type_code', y='passenger_count')
plt.title('Distribución de Pasajeros por Tipo de Vuelo')
plt.show()

# Fase 2: Segmentación de Aerolíneas
En esta fase, utilizaremos técnicas de clustering para agrupar aerolíneas en categorías operativas distintas.

In [ ]:
# Ingeniería de características
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Crear características agregadas
features = data.groupby('operating_airline').agg({
    'passenger_count': ['sum', 'mean'],
    'geo_region': 'nunique',
    'activity_type_code': lambda x: (x == 'International').mean()
})
features.columns = ['total_passengers', 'avg_passengers', 'unique_regions', 'intl_operations']

# Escalado de características
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [ ]:
# Método del Codo para determinar el número óptimo de clústeres
inertia = []
for k in range(1, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 10), inertia, marker='o')
plt.title('Método del Codo')
plt.xlabel('Número de Clústeres')
plt.ylabel('Inercia')
plt.show()

In [ ]:
# Aplicar K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
features['cluster'] = kmeans.fit_predict(scaled_features)

# Analizar los centroides
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=features.columns[:-1])
print(centroids)

# Fase 3: Predicción de Volumen de Pasajeros
Construiremos un modelo de regresión para predecir el número de pasajeros por vuelo.

In [ ]:
# Preparación de datos para regresión
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Selección de características y variable objetivo
X = pd.get_dummies(data[['operating_airline', 'geo_region', 'terminal']])
y = data['passenger_count']

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar el modelo
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Evaluar el modelo
y_pred = model.predict(X_test)
print('RMSE:', mean_squared_error(y_test, y_pred, squared=False))
print('R²:', r2_score(y_test, y_pred))

# Fase 4: Pronóstico de Tráfico Aéreo Mensual
Utilizaremos modelos de series temporales para pronosticar la demanda futura de pasajeros.

In [ ]:
# Crear serie temporal
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Agregar pasajeros por mes
time_series = data.groupby(data['date'].dt.to_period('M'))['passenger_count'].sum()
time_series.index = time_series.index.to_timestamp()

# Descomposición de la serie
decomposition = seasonal_decompose(time_series)
decomposition.plot()
plt.show()

In [ ]:
# Entrenar modelo SARIMA
model = SARIMAX(time_series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
results = model.fit()

# Pronóstico
forecast = results.get_forecast(steps=24)
forecast_index = pd.date_range(start=time_series.index[-1], periods=24, freq='M')
forecast_values = forecast.predicted_mean

# Visualizar pronóstico
plt.plot(time_series, label='Histórico')
plt.plot(forecast_index, forecast_values, label='Pronóstico', color='red')
plt.fill_between(forecast_index, forecast.conf_int()['lower passenger_count'], forecast.conf_int()['upper passenger_count'], color='pink', alpha=0.3)
plt.legend()
plt.show()

# Fase 5: Presentación de Resultados
Consolidaremos los hallazgos y crearemos visualizaciones avanzadas para comunicar los resultados.

In [ ]:
# Crear visualizaciones clave
import plotly.express as px

# Gráfico de segmentos de aerolíneas
fig = px.bar(features, x=features.index, y='total_passengers', color='cluster', title='Segmentos de Aerolíneas')
fig.show()

# Mapa de rutas más populares
# (Requiere datos adicionales de coordenadas geográficas)